In [1]:
import pandas as pd

### 1) Load the worklist

In [2]:
# read the worklist exported from mysql
wl = pd.read_csv("../output/collection_worklist.csv")

# priority 1 first
wl = wl.sort_values("priority").reset_index(drop=True)

wl[["priority", "invoice_id", "clearer_name", "open_eur", "days_past_due", "next_action"]]

,priority,invoice_id,clearer_name,open_eur,days_past_due,next_action
0,1,INV-202609-HAL,Halcyon Bank,15733.00,10,"Apply unreferenced payment, then close"
1,2,INV-202607-TAR,Tamsin & Rowe,697.86,69,Resolve rate dispute with Sales / Client Manag...
2,3,INV-202608-MER,Meridian Capital Markets,13806.50,40,Escalate: payment promise broken
3,3,INV-202608-VAL,Valemont Bank,3248.75,40,Escalate: payment promise broken
4,4,INV-202609-MER,Meridian Capital Markets,13493.50,10,Call clearer
5,5,INV-202609-VAL,Valemont Bank,5723.00,10,Wait: promise not yet due


### 2) The email writing function

In [3]:
def draft_email(row):
    inv = row["invoice_id"]
    name = row["clearer_name"]
    amount = f"EUR {row['open_eur']:,.2f}"
    days = row["days_past_due"]
    promised = row["latest_promised_date"]

    if row["priority"] == 1:
        to = "Internal: Finance"
        subject = f"Apply unreferenced payment {row['matching_payment']} to {inv}"
        body = (
            f"Payment {row['matching_payment']} from {name} ({amount}) arrived without an invoice reference.\n"
            f"Clearer and amount match open invoice {inv}.\n"
            f"Please confirm and apply the payment, then close the item. No chaser needed."
        )
    elif row["priority"] == 2:
        to = "Internal: Sales / Client Management"
        subject = f"Rate dispute on {inv} ({name}) - please confirm rates"
        body = (
            f"{name} has short-paid {inv}; {amount} is still open ({days} days overdue).\n"
            f"They say the rates billed differ from their schedule.\n"
            f"Please confirm the agreed rates so we can either re-bill or request the balance."
        )
    elif row["priority"] == 3:
        to = name
        subject = f"Overdue: {inv} - payment promised for {promised} not received"
        body = (
            f"Dear {name} team,\n\n"
            f"On our last call you confirmed full payment of {inv} by {promised}.\n"
            f"As of today the full payment has not been received, and {amount} remains outstanding ({days} days overdue).\n"
            f"Please arrange payment this week or let us know what is blocking it.\n\n"
            f"Kind regards,\nTrading Operations - Accounts Receivable"
        )
    elif row["priority"] == 4:
        to = name
        subject = f"Payment reminder: {inv} ({days} days overdue)"
        body = (
            f"Dear {name} team,\n\n"
            f"Our records show {amount} open on invoice {inv}, now {days} days past due.\n"
            f"Could you confirm when we can expect payment?\n\n"
            f"Kind regards,\nTrading Operations - Accounts Receivable"
        )
    else:
        to = "No email"
        subject = f"Hold: {name} promised payment of {inv} by {promised}"
        body = f"No contact until {promised}. Re-check the payment on that date."

    return {"invoice_id": inv, "to": to, "subject": subject, "body": body}


# test on the first row only
print(draft_email(wl.iloc[0]))

{'invoice_id': 'INV-202609-HAL', 'to': 'Internal: Finance', 'subject': 'Apply unreferenced payment PAY-0011 to INV-202609-HAL', 'body': 'Payment PAY-0011 from Halcyon Bank (EUR 15,733.00) arrived without an invoice reference.\nClearer and amount match open invoice INV-202609-HAL.\nPlease confirm and apply the payment, then close the item. No chaser needed.'}


### 3) Drafting all emails and showing them readably

In [4]:
# draft an email for every row
drafts = []
for i in range(len(wl)):
    row = wl.iloc[i]
    drafts.append(draft_email(row))

emails = pd.DataFrame(drafts)

# print each draft in a readable layout
for i in range(len(emails)):
    e = emails.iloc[i]
    print("=" * 70)
    print("TO:     ", e["to"])
    print("SUBJECT:", e["subject"])
    print()
    print(e["body"])

TO:      Internal: Finance
SUBJECT: Apply unreferenced payment PAY-0011 to INV-202609-HAL

Payment PAY-0011 from Halcyon Bank (EUR 15,733.00) arrived without an invoice reference.
Clearer and amount match open invoice INV-202609-HAL.
Please confirm and apply the payment, then close the item. No chaser needed.
TO:      Internal: Sales / Client Management
SUBJECT: Rate dispute on INV-202607-TAR (Tamsin & Rowe) - please confirm rates

Tamsin & Rowe has short-paid INV-202607-TAR; EUR 697.86 is still open (69 days overdue).
They say the rates billed differ from their schedule.
Please confirm the agreed rates so we can either re-bill or request the balance.
TO:      Meridian Capital Markets
SUBJECT: Overdue: INV-202608-MER - payment promised for 2026-10-20 not received

Dear Meridian Capital Markets team,

On our last call you confirmed full payment of INV-202608-MER by 2026-10-20.
As of today the full payment has not been received, and EUR 13,806.50 remains outstanding (40 days overdue).
Pl

### 4) Save the drafts

In [5]:
# save the drafts
emails.to_csv("../output/collection_emails.csv", index=False)
print(len(emails), "drafts saved")

6 drafts saved
